# Interpolation and Fitting

This week we are focussing on the method of interpolation and fitting.  These are related in that both can involve taking a sparsely sampled set of data and trying to choose or create a function that approximate the data.

We usually use **interpolation** algorithms when we either know the functional form of the underlying data or we don't necessarily care in detail about this functional form and simply need an approximation at intermediate points.

We usually use **fitting** algorithms when we are uncertain about the underlying process that created some data and we want attempt to model it with some parameterized functional form.  In this case, we usually have some fit statistic that we are trying to minimize to choose the best underlying model or the best parameterization of the model.  Since we usually minimize this fit statistic, algorithms for fitting are closely related to the optimization algorithms we discussed last week.

## 1 Interpolation

Interpolation is common tool used in computational science.  In arises in many contexts, and underlies several numerical algorithms. For example, we encountered it last week in the form quadratic interpolation while discussing root finding and optimization algorithms.

A general problem is the following: you are given the value of a function $f(x)$ at a discrete number of points $x_i$ but you want to know the value at some point (or points) $x$ between the sampled points, but you do not know the precise form of $f(x)$ or it is to expensive to evaluate.  What do you do?

There are a number of possibilities.  Lets start by considering a simple example: just two points $x = x_i, x_{i+1}$ and you want to know the value of the function at another point $x$ in between. 

In [ ]:
# Let's start with a simple example
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# lets use a sin function to generate the points
f = lambda x: np.sin(x)

# next generate some evenly sampled points
x_min, x_max = 0, np.pi
npt0 = 100
npt1 = 4

x0 = np.linspace(x_min, x_max, npt0)
x1 = np.linspace(x_min, x_max, npt1) 

plt.plot(x0,f(x0),label='true function')
plt.scatter(x1,f(x1), color='r', label='available points')

plt.legend()
plt.xlabel('x')
plt.ylabel('sin(x)')
plt.xlim([0,np.pi]);

### Linear interpolation

The simplest interpolation is linear interpolation.  If all you care about is make sure the estimate is between the points, then this is the easiest thing to do.

We assume our function follows a straight line from $y_i=f(x_i)$ to $y_{i+1}=f(x_{i+1})$, which in most cases is an approximation—likely the true function follows a curve between the two points, as in the figure above. But if we make this assumption then we can esimtate $y= f(x)$ without much work.

The slope of the straight-line approximation is
\begin{equation}
m = \frac{y_{i+1}-y_{i}}{x_{i+1}-x_i}
\end{equation}
We then have
\begin{equation}
y = y_i+m(x-x_i)
\end{equation}
or, equivalently
\begin{equation}
y = y_{i+1}-m(x_{i+1}-x)
\end{equation}

In [ ]:
def interpolate_linear(x, y, xint):
    """Perform linear interpolation
    inputs:
    x (ndarray): abscissas where functions are defined
    y (ndarray): values at the abscissas 
    xint (ndarray): abscissas on which to interpolate

    returns:
    yint (ndarray) interpolated values
    """
    # compute slopes
    m = (y[1:] - y[:-1]) / (x[1:] - x[:-1])

    # assumes xint values are sorted lowest to highest
    # check to make sure bounds are satisfied
    if xint[0] < x[0]:
        print("Error xint[0]: ",xint[0]," < x[0]: ",x[0])
    if xint[-1] < x[-1]:
        print("Error xint[-1]: ",xint[0]," > x[-1]: ",x[0])
    i = 0
    yint = np.empty_like(xint)
    for j,xi in enumerate(xint):
        # ensure xi is between x[i] and x[i+1]
        while (xi < x[i]) or (xi > x[i+1]):
            i += 1
        # compute interpolated value
        yint[j] = y[i] + m[i] * (xi-x[i])

    return yint

In [ ]:
# Let's start with a simple example
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

f = lambda x: np.sin(x) 
x_min, x_max = 0, np.pi
npt0 = 100 # true function
npt1 = 4   # sampled points
x0 = np.linspace(x_min, x_max, npt0) # for true function
x1 = np.linspace(x_min, x_max, npt1) # available points
y1 = f(x1) # values at available points

# interpolate on to grid with 10 pts for every known point
npt2 = 10
x2 = np.linspace(x_min, x_max, npt1*npt2)

# perfrom interpolation
y2 = interpolate_linear(x1, y1, x2)

#plot all points
plt.plot(x0,f(x0),label='true function')
plt.scatter(x1,y1, color='r', label='sampled points')
plt.plot(x2,y2, '*b', label='linear interpolation')

plt.legend()
plt.xlabel('x')
plt.ylabel('f(x)')
plt.xlim([x_min, x_max]);

Although this is simple, we can see that linear interpolation might not be a good approximation when the true function is far from being linear. This could introduce significant errors in some applications. An obvious mode for improvement is to try higher order interpolating polynomials.

### Spline interpolation

When the function itself is not linear, it is often (but not always) a better approximation is to use polynomial functions. **Spline interpolation** is a form of interpolation where the interpolant is a special type of piecewise polynomial called a spline. 

**Linear interpolation**, in this context, is just the simplest version of spline interpolation.
\begin{equation}
S_i(x) = y_i + \frac{y_{i+1}-y_i}{x_{i+1}-xi}(x-x_i)
\end{equation}

The next higher order method is **quardratic spline interpolation**. The relation is a bit more complicated, so let's start with a simple example.

Assume we have three points $(x_i,y_i)$: $(-1,0)$, $(0,1)$, $(1,3)$

These define to intervals: -1 to 0 and 0 to 1. We want to find the quadratic spline:

\begin{equation}
p_1(x) = a_1 + b_1x + c_1x^2 \qquad [-1,0]
\end{equation}
\begin{equation}
p_2(x) = a_2 + b_2x + c_2x^2 \qquad [0,1]
\end{equation}
They should satisfy the following conditions
\begin{eqnarray}
p_1(-1) & = & a_1-b_1+c_1 = 0,\\
p_1(0) & = & a_1 = 1,\\
p_2(0) & = & a_2 = 1,\\
p_2(1) & = & a_2 + b_2 + c_2 = 3
\end{eqnarray}

In addition, we want to curves to be continous, therefore $p_1(0)' = p_2(0)'$ gives
\begin{equation}
b_1 = b_2
\end{equation}
There are 6 variables in total, but we only have five equations, so one must define another condition.
Usually, we can impose the constrain about the derivative of the ending members, say let $p_1(-1)'$=0,
\begin{equation}
b_1 - 2c_1 = 0
\end{equation}

Therefore, we can solve the equations, and get all coefficients for the quadratic spline function:

\begin{eqnarray}
p_1(x) & = & 1 + 2x + x^2 &\qquad [-1,0]\\
p_2(x) & = & 1 + 2x  & \qquad [0,1]
\end{eqnarray}

If we do interpolation for a series of points, the generallized relation is the following,
\begin{equation}
S_i(x) = y_i + z_i(x-x_i) + \frac{z_{i+1}-z_i}{2(x_{i+1}-x_i)}(x-x_i)^2
\end{equation}
where the $z$ series can be obtained via
\begin{equation}
z_{i+1} = -z_i + 2\frac{y_{i+1}-y_i}{x_{i+1}-x_i}
\end{equation}

and $z_0$ is the derivative of the 1st point.

In [ ]:
def spline_quadratic(x, y, xint):
    """Perform quadratic spline interpolation
    inputs:
    x (ndarray): abscissas where functions are defined
    y (ndarray): values at the abscissas 
    xint (ndarray): abscissas on which to interpolate

    returns:
    yint (ndarray) interpolated values
    """
    z = np.empty_like(x)
    z[0] = 1 # one can try other values here
    # compute slopes and zs
    m = (y[1:] - y[:-1]) / (x[1:] - x[:-1])
    
    for i in range(1,z.shape[0]):
        z[i] = -z[i-1] + 2 * m[i-1]

    # assumes xint values are sorted lowest to highest
    # check to make sure bounds are satisfied
    if xint[0] < x[0]:
        print("Error xint[0]: ",xint[0]," < x[0]: ",x[0])
    if xint[-1] > x[-1]:
        print("Error xint[-1]: ",xint[0]," > x[-1]: ",x[0])
    i = 0
    yint = np.empty_like(xint)
    for j,xi in enumerate(xint):
        # ensure xi is between x[i] and x[i+1]
        while (xi < x[i]) or (xi > x[i+1]):
            i += 1
        # compute interpolated value
        delta = xi - x[i]
        yint[j]=y[i]+z[i]*delta+(z[i+1]-z[i])/2/(x[i+1]-x[i])*delta**2

    return yint

In [ ]:
# Let's plot an example
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

f = lambda x: np.sin(x) 
x_min, x_max = 0, np.pi
npt0 = 100 # true function
npt1 = 4   # sampled points
x0 = np.linspace(x_min, x_max, npt0) # for true function
x1 = np.linspace(x_min, x_max, npt1) # available points
y1 = f(x1) # values at available points

# interpolate on to grid with 10 pts for every known point
npt2 = 10
x2 = np.linspace(x_min, x_max, npt1*npt2)

# perfrom interpolation
y2 = spline_quadratic(x1, y1, x2)

#plot all points
plt.plot(x0,f(x0),label='true function')
plt.scatter(x1,y1, color='r', label='sampled points')
plt.plot(x2,y2, '*b', label='linear interpolation')

plt.legend()
plt.xlabel('x')
plt.ylabel('f(x)')
plt.xlim([x_min,x_max]);

### General Discussion

For a curve to be continuous, we have to make sure each segment gives the equal value when they meet

\begin{equation}
f_i(x_i) = f_{i+1}(x_i)
\end{equation}
This is what linear spline does.

In order to make the curve smooth, we also ask that the 1st derivatives to be equal

\begin{equation}
(f_i(x_i))' = (f_{i+1}(x_i))'
\end{equation}

which has been used in the quadratic spline interpolation above.

For the segments to have the same curvature when they meet, we impose
\begin{equation}
(f_i(x_i))'' = (f_{i+1}(x_i))''
\end{equation}
One has to go to **cubic spline** interpolation in order to satisfy this condition.

Cubic spline is the most popular choice when most people do interpolation.  It has smooth curvature and a good tradeoff in computational cost. Generalization the approach above for quadratic splines yields:
\begin{equation}
S_i(x) = \frac{z_{i+1}(x-x_i)^3 + z_i(x_{i+1}-x)^3} {6h_i} + \left(\frac{y_{i+1}}{h_i}-\frac{h_i}{6}z_{i+1}\right) + \left(\frac{y_i}{h_i}-\frac{h_i}{6}z_i\right) (x_{i+1}-x_i)
\end{equation}
where $h_i = x_{i+1} - x_i$, and the $z$ series can be obtained via
\begin{equation}
h_{i-1}z_{i-1} + 2(h_{i-1}+h_i)z_i + h_iz_{i+1} =6\left(\frac{y_{i+1}}{h_i}-\frac{y_i-y_{i-1}}{h_{i-1}}\right)
\end{equation}

### Interpolation with SciPy

Although it is good to try coding up algorithms from scratch to learn them, the expressions start to become tedious to implement.  This is starting to be the case even for cubic spline interpolation.  Fortunately, SciPy has provided useful tools for interpolation:

https://docs.scipy.org/doc/scipy/reference/interpolate.html

I have generally used the `scipy.interpolate.interp1d()` function but this is now marked as a legacy method.  For 1d linear interpolation there the NumPy routine `interp()`, which is probably faster:

https://numpy.org/devdocs/reference/generated/numpy.interp.html#numpy.interp

There is also `scipy.interpolate.CubicSplines()` for implementing cubic spline interpolation as well as a large number of types of spline interpolation.

In [ ]:
# interpolation with scipy
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import scipy.interpolate as interpolate

f = lambda x: np.sin(x) 
x_min, x_max = 0, 2*np.pi
npt0 = 100 # true function
npt1 = 4   # sampled points
x0 = np.linspace(x_min, x_max, npt0) # for true function
x1 = np.linspace(x_min, x_max, npt1) # available points
y1 = f(x1) # values at available points

# interpolate on to grid with 10 pts for every known point
npt2 = 10
x2 = np.linspace(x_min, x_max, npt1*npt2)

# perfrom interpolation
f_spline1 = interpolate.interp1d(x1, y1, kind='linear')
f_spline2 = interpolate.interp1d(x1, y1, kind='quadratic')
f_spline3 = interpolate.CubicSpline(x1, y1)

#plot all points
plt.plot(x0,f(x0),label='true function')
plt.plot(x2,f_spline1(x2), '--',  label='linear')
plt.plot(x2,f_spline2(x2), '-', label='quadratic')
plt.plot(x2,f_spline3(x2), '-.', label='cubic')
plt.scatter(x1,f(x1), c='r', s=180, marker='*', label='available points')

plt.legend()
plt.xlabel('x')
plt.ylabel('f(x)')
plt.xlim([x_min, x_max]);

### Going to Higher Order

You may ask if going to even higher order than cubic is better?  Maybe? It really depends on the problem, but there are a number of tradeoffs to consider and the choice will depend on your goals.
* Higher order is more expensive and complex
* Higher order is not always more accurate
* Higher order can introduce new minima and maxima (wiggles) which may present problems for some algorithms, such as when using interpolation in solving non-linear partial differential equations.

Sometimes higher order interpolation is worth it.  One just needs to be careful in how one uses it.

### Coefficients of the Interpolating Polynomial

Sometime one would like to explicitly solve for the coefficients of the polynomials to be used in the interpolation. Notice that in the above examples we didn't do this.  It is often easier (and more accurate) to just do the algebra and construct the expression for the interpolant directly. There are, however, cases where knowing the coefficients can be useful. If you want to take derivatives or integral of the function, it is very easy to code up expression for the integrals or derivatives of the polynomial approximating the function if you know the coefficients.

Consider the case where we have $n$ tabulations $(x_i,y_i)$. We can determine a polynomial of the form
\begin{equation}
y = c_0 + c_1 x + c_2 x^2 + \dots + c_{n-1} x^{n-1}
\end{equation}
This polynomial goes through all of the points and is called the Lagrange polynomial.
We can solve for the coefficients $c_i$ via
\begin{equation}
\begin{pmatrix}
1 & x_0 & x_0^2 & \dots & x_0^{n-1}\\
1 & x_1 & x_1^2 & \dots & x_1^{n-1}\\
1 & x_2 & x_2^2 & \dots & x_2^{n-1}\\
\vdots & \vdots & \vdots & \ddots &\vdots \\
1 & x_{n-1} & x_{n-1}^2 & \dots & x_{n-1}^{n-1}
\end{pmatrix}
\begin{pmatrix}
c_0\\
c_1\\
c_2\\
\vdots\\
c_{n-1}
\end{pmatrix}
=
\begin{pmatrix}
y_0\\
y_1\\
y_2\\
\vdots\\
y_{n-1}
\end{pmatrix}
\end{equation}
This is called the Vandermonde matrix. Directly inverting the matrix is $O(n^3)$.  There are more efficient algorithms for solving this but for illustration we can just use `numpy.linalg` to invert the matrix

## **Student Complettion**

Complete the function for constructing the Vandermonde matrix

In [ ]:
def vandermonde_matix(x):
    """Construct Vandermode matrix"""
    n = x.shape[0]
    v = np.zeros([n,n])
    for i in range(n):
        for j in range(n):
            # your code here
    return v

def evaluate_polynomial(x, c):
    """Evaluate polynomial given coefficients"""

    y = np.zeros_like(x)
    for j in range(c.shape[0]):
        y += c[j]*x**j
    return y

In [ ]:
#MINE
def vandermonde_matix(x):
    """Construct Vandermode matrix"""
    n = x.shape[0]
    v = np.zeros([n,n])
    for i in range(n):
        for j in range(n):
            v[i,j] = x[i]**j
    return v

def evaluate_polynomial(x, c):
    """Evaluate polynomial given coefficients"""

    y = np.zeros_like(x)
    for j in range(c.shape[0]):
        y += c[j]*x**j
    return y

In [ ]:
# lets use this to construct some solutions
%matplotlib inline
f = lambda x: np.exp(-x**2) 
x_min, x_max = -1, 1
npt0 = 100 # true function
x0 = np.linspace(x_min, x_max, npt0) # for true function
npt1 = 4
x1 = np.linspace(x_min, x_max, npt1) # available points
y1 = f(x1) # values at available points

# solve for coefficients
v = vandermonde_matix(x1)
vinv = np.linalg.inv(v)
coeff4 = vinv @ y1
yint = evaluate_polynomial(x0, coeff4)

#plot all points
plt.plot(x0,f(x0),label='true function')
plt.scatter(x1,y1, color='r', label='sampled points')
plt.plot(x0, yint, '.b', label='interpolating polynomial')
plt.legend();

What happens if we use eight sampled points instead of four?

In [ ]:
%matplotlib inline
npt1 = 8 # only change from above

f = lambda x: np.exp(-x**2) 
x_min, x_max = -1, 1
npt0 = 100 # true function
x0 = np.linspace(x_min, x_max, npt0) # for true function

x1 = np.linspace(x_min, x_max, npt1) # available points
y1 = f(x1) # values at available points

# solve for coefficients
v = vandermonde_matix(x1)
vinv = np.linalg.inv(v)
coeff8 = vinv @ y1
yint = evaluate_polynomial(x0, coeff8)

#plot all points
plt.plot(x0,f(x0),label='true function')
plt.scatter(x1,y1, color='r', label='sampled points')
plt.plot(x0, yint, '.b', label='interpolating polynomial')
plt.legend();

This looks much better. So we get a good representation of the function, but this is really not the advantage of this approach, which could have been achieved more cheaply with other spline methods.

The key advantage is that we now have a representaion that is very easy to differentiate or integrate.  For example, lets consider integration.

In [ ]:
def integrate_polynomial(xmin, xmax, c):
    """integrate a polynomial"""
    
    j1 = np.arange(c.shape[0])+1
    sum1 = np.sum(c*xmin**j1/j1)
    sum2 = np.sum(c*xmax**j1/j1)
    
    return sum2-sum1

In [ ]:
from scipy.special import erf

# compute exact solution of integral
exact = (np.sqrt(np.pi)/2.0) * (erf(1.0) - erf(-1.0))
int4 = integrate_polynomial(x_min, x_max, coeff4)
int8 = integrate_polynomial(x_min, x_max, coeff8)
print("Exact solution: ", exact)
print("Integral, relative error: ",int4, (int4-exact)/exact)
print("Integral, relative error: ",int8, (int8-exact)/exact)

We observer that we get fast convergence to the true solution as we increase the number of points used to sample the solution.

Here we used uniformly distributed abscissas for samples. Recall from our discussion of quadratures that we can often do better by choosing the abscissas.  We specifically considered abscissas that were set by the zeros of the Legendre poylynomials to perform Guass-Legendre quadrature.  We can do the same here.

In [ ]:
from scipy.special import roots_legendre

# compute abscissas and weights for Gauss-Legendre
x4, w4 = roots_legendre(4)
x8, w8 = roots_legendre(8)

# construct polynomial and perform integral
# with 4 points
v = vandermonde_matix(x4)
vinv = np.linalg.inv(v)
coeff4 = vinv @ f(x4)
int4 = integrate_polynomial(x_min, x_max, coeff4)

# construct polynomial and perform integral
# with 8 points
v = vandermonde_matix(x8)
vinv = np.linalg.inv(v)
coeff8 = vinv @ f(x8)
int8 = integrate_polynomial(x_min, x_max, coeff8)
print("Exact solution: ", exact)
print("Integral, relative error: ",int4, (int4-exact)/exact)
print("Integral, relative error: ",int8, (int8-exact)/exact)

We get even better results as expected.  How does this compare to using the Gauss-Legendre quadrature directly?

In [ ]:
# perform integrals with Gauss-Legendre quadrature
int4 = np.sum(f(x4)*w4)
int8 = np.sum(f(x8)*w8)

print("Exact solution: ", exact)
print("Integral, relative error: ",int4, (int4-exact)/exact)
print("Integral, relative error: ",int8, (int8-exact)/exact)

It turns out that integration with our interpolating polynomial expression corresponds exactly to Gauss-Legendre quadrature if the abscissa are the same.

We can get similar results for performing derivatives as well.  This functionality can be very useful when the function being interpolated is expensive to calculate direcltly e.g. it is the result of some complicated numerical calcualtion and not a simple analytic function.

## 2 Fitting

As mentioned above, fitting and interpolation are superficially related in that they both try to construct representations of sampled data. But, in costrast to interpolation, fitting is usually an attempt to ascribe a parameterized model to the sampled data. The fitting excercise can include both trying to find the optimal parameters that reproduce the data and to assess how well the model corresponds to the sampled data. This fitting procedure is performed relative to some metric (a fit statistic) that evaluates the match of the model to the data. Hence, fitting is usually an optimization problem that minimizes or maximizes a fit statistic.

It often assumed that the data has some measurment error in the ordinate, and possibly the abscissa as well. Hence, the fit does not try to exactly match the sampled data (as is usually the case in interpolation).

### Polynomial Fitting

We already encountered this problem in the homework from last week, where we considered two different fit statistics.  Lets first consider the least squares statistic here.  

Lets assume we have $n$ pairs of abscissas and ordinates

$(x_0, y_0), (x_1, y_1), (x_2, y_2),  ... ,(x_{n-1}, y_{n-1})$

We want to determine a linear relation $f(x) = ax+b$ that provides the "best" description of these points. What should our criterion be?

One possible estimate the error is
\begin{equation}
S(\mathbf{\beta}) = \sum_{i=0}^{n-1}\left[y_i - f(x_i,\mathbf{\beta})\right]^2.
\end{equation}
For this linear relation, we have $\mathbf{\beta}=(a, b)$, and
\begin{equation}
S = \sum_{i=0}^{n-1}\left[y_i - (ax_i + b)\right]^2
\end{equation}
The best line has minimum error between line and data points.

This is called the *least squares approach*, the goal is to find the minimum value of the fit statistic $S$.

To slove it, we simply just need to find the point where 
\begin{equation}
\frac{\partial{S}}{\partial a} = -2\sum_{i=0}^{n-1} x_i (y_i - ax_i -b) = 0
\end{equation}
and 
\begin{equation}
\frac{\partial{S}}{\partial b} = -2\sum_{i=0}^{n-1} (y_i - ax_i -b) = 0
\end{equation}
These two equations could be rewritten as 
\begin{equation}
a\sum x_i^2 + b\sum x_i = \sum(x_iy_i),
\end{equation}
and
\begin{equation}
a\sum x_i + bn = \sum{y_i},
\end{equation}
where the sum runs from $i=0$ to $n-1$.

We can write this in matrix form as

\begin{equation}
\begin{pmatrix}
       n     & \sum x_i    \\
    \sum x_i & \sum x_i^2  \\
\end{pmatrix}
\begin{pmatrix} 
       b \\
       a \\
\end{pmatrix}
=
\begin{matrix}
      \sum y_i \\
      \sum (x_iy_i)\\
\end{matrix}
\end{equation}

So this is basically a problem to solve Ax = B

For polynomial fitting, this scheme could be generalized for a polynomial of order $m$ as 
    
\begin{equation}
\begin{pmatrix}
       n       & \sum x_i       & \sum x_i^2     & \dots  & \sum x_i^m     \\
    \sum x_i   & \sum x_i^2     & \sum x_i^3     & \dots  & \sum x_i^{m+1} \\
    \sum x_i^2 & \sum x_i^3     & \sum x_i^4     & \dots  & \sum x_i^{m+2} \\
    \vdots     & \vdots         & \vdots         & \ddots & \vdots \\
    \sum x_i^m & \sum x_i^{m+1} & \sum x_i^{m+2} & \dots  & \sum x_i^{2 m} 
\end{pmatrix}
\begin{pmatrix}
    a_0   \\
    a_1   \\
    a_2   \\
    \vdots  \\
    a_m 
\end{pmatrix}
=
\begin{pmatrix}
    \sum y_i   \\
    \sum (x_iy_i)   \\
    \sum (x_i^2y_i) \\
    \vdots \\
    \sum (x_i^my_i) 
\end{pmatrix}
\end{equation}

We could now apply the algorithms for solving systems of equations to this matrix to solve for $a_i$.  For polynomial fitting, NumPy provides the routines `polyfit()` and `polyval()`.

In [ ]:
import numpy as np

#define a function for producing the data
f = lambda x: x * np.sin(x)  

#define the paramters for the plot
x_min, x_max = 0, 3
npoints = 100

# generate plot data with random deviations
x = np.linspace(x_min, x_max, npoints)
y = f(x) + 0.5*(np.random.rand(npoints) - 0.5)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=[12, 8])

# fit the polynomial
order = 5 # try different values
fit = np.polyfit(x, y, order, full=True)
yfit = np.polyval(fit[0], x)

plt.plot(x, f(x), '--', label='true function')
plt.plot(x, y, 'o', label='data')
plt.plot(x, yfit, label='polyfit '+str(order))

plt.xlabel('x')
plt.ylabel('$f(x)$')
plt.xlim([x_min, x_max])
plt.legend(fontsize=12)
print('fitted coefficients: ' + str(fit[0]) + '\nresidual: ' + str(fit[1]))

## **Student Completion**

Try to run the code with a different polynomial order and determine which polynomial yields the best fit (lowest residual) for values between 1 and 10.  Does the fit always get better with higher order?

### General Fitting

We can also fit a general function $f(x, \mathbf{\beta})$ where $\beta$ represents a vector of an arbitrary number of parameters. On last week's homework, you were supposed to come up with a function to do this using `scipy.minimize()`.  An example implementation follows.

In [ ]:
from scipy.optimize import minimize

def least_squares_fit(x, y, model, guess):
    """
    Perform least squares fitting using scipy.optimize.minimize.

    Parameters:
    x (numpy array): Independent variable.
    y (numpy array): Dependent variable.
    model : Model function to fit the data.
    guess (numpy array): Initial guess for the parameters of the model.

    returns the result
    """

    # Define the fit statistic as the function to be minimized
    def statistic(params):
        return np.sum((y - model(x, *params))**2)

    #  Minimize the least squares statistic
    res = minimize(statistic, guess)

    return res.x

Lets try it out and compare to the results from `scipy.optimize.curve_fit()`.

In [ ]:
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=[10, 6])

#define the function
f = lambda x, a, b, c: a*np.exp(-b*x)+c  

#define the paramters for the plot
x_min, x_max = 0, 10
npoints = 100
a, b, c = 2.5, 1.3, 0.4

x = np.linspace(x_min, x_max, npoints)
y = f(x, a, b, c) + 0.5*(np.random.rand(npoints) - 0.5)

# fit with curve_fit
params, pcov = curve_fit(f, x, y)

# fit with least_squares_fit
guess = [1, 1, 1]
res = least_squares_fit(x, y, f, guess)

plt.plot(x, f(x,a,b,c), '--', label='true function')
plt.plot(x, y, 'o', label='data')
plt.plot(x, f(x,params[0],params[1],params[2]), label='curve_fit')
plt.plot(x, f(x,res[0],res[1],res[2]), label='least_squares_fit')

plt.xlabel('x')
plt.ylabel('$f(x)$')
plt.xlim([x_min, x_max])
plt.legend(fontsize=12)
plt.show()
print('original coefficients: %6.3f, %6.3f, %6.3f' %(a,b,c))
print('curve_fit coefficients: %6.3f, %6.3f, %6.3f' %(params[0], params[1], params[2]))
print('least_squares_fit coefficients: %6.3f, %6.3f, %6.3f' %(params[0], params[1], params[2]))

We see that our implementation yields the same result because this is essentially what the `curve_fit()` function does. It defines a least squares fitting statistic and the uses SciPy's optimization routines to minimize it.

### Estimating Errors and the Covariance Matrix

One important thing that `curve_fit()` does that is not accounted for in our `least_squares_fit()` is that is also outputs the covariance matrix. This information is very useful if you want to get a sense of uncertainty on the parameters.  The (square root) of the diagonal elements of the covariance matrix provide a sense of the uncertainty in the best fit parameter, while the off-diagonal component provide an indication of the (anti)correlation between any two parameters.

In [ ]:
# Lets use the covariance matrix to estimate
# an uncertainty on model parameter
%matplotlib inline
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=[10, 6])

#define a sinusoidal function
f2 = lambda x, a, b, c: a*np.sin(b*x+c)  

#define the paramters for the plot
x_min, x_max = 0, 2.*np.pi
npoints = 50
a, b, c = 2.5, 1.3, 0.4

x = np.linspace(x_min, x_max, npoints)
# generate random noise with a normal distribution
amp = 0.5
rng = np.random.default_rng()
delta = amp*rng.normal(size=(x.shape[0],))
y = f2(x, a, b, c) + delta

# give each point the same error estimate
yerr = np.ones_like(x)*delta.std() 

# fit with curve_fit
params, pcov = curve_fit(f2, x, y, sigma=yerr)

# use covariance matrix to estimate error on range
# of param[2]
p2max = params[2] + np.sqrt(pcov[2,2])
p2min = params[2] - np.sqrt(pcov[2,2])

plt.errorbar(x,y,yerr,ls='none', marker='o', label='data')
plt.plot(x, f2(x,params[0],params[1],p2max), label='max')
plt.plot(x, f2(x,params[0],params[1],p2min), label='min')

plt.xlabel('x')
plt.ylabel('$f(x)$')
plt.xlim([x_min, x_max])
plt.legend(fontsize=12);

print('original coefficients: %6.3f, %6.3f, %6.3f' %(a, b, c))
print('curve_fit coefficients: %6.3f, %6.3f, %6.3f' %(params[0], params[1], params[2]))
print('param[2] min, max: %6.3f, %6.3f' %(p2min, p2max))

In practice, this is not the best way to estimate errors on fit parameters.  For normally disributed data, if are confident the model provides a good fit to the data, we can usually assume that changes in the underlying parameters will follow a $\chi^2$ distirbution and define confidence intervals based on the change in $\chi^2$ that occurs from varying a single parameter (and keeping all others fixed) or joint confidence intervals that come from changing multiple parameters (and keeping all others fixed).

Recall that the $\chi^2$ statistic is computed via
\begin{equation}
\chi^2 =\sum_{i=1}^{n} [\frac{y_i - f(x_i, \mathbf{\beta})}{\sigma_i}]^2.
\end{equation}

## **Student Completion**

Compute the $\chi^2$ statistic

In [ ]:
# evaluate the chi**2 for our best fit params
def chi2_statistic(x, y, yerr, f, params):
    pass

In [ ]:
# Mine
# evaluate the chi**2 for our best fit params
def chi2_statistic(x, y, yerr, f, params):
    resid = (y - f(x, *params)) / yerr
    return np.sum(resid**2)

For a single parameter, $\Delta \chi^2 = 2.706$ corresponds to a 90% confidence interval.

In [ ]:
from scipy.optimize import root_scalar

def dchi2(val, ind, lev, f, chi2min):

    def ffixed(x, a, b):
        # keep it simple
        if ind == 0:
            return f(x, val, a, b)
        elif ind == 1:
            return f(x, a, val, b)
        elif ind == 2:
            return f(x, a, b, val)
    # repeat fit with on value fixed
    p, pcov = curve_fit(ffixed, x, y, sigma=yerr)
    # lev: delta chi**2 = 2.706 is 90% confidence for one parameter
    p = np.insert(p, ind, val)
    return chi2_statistic(x, y, yerr, f2, p)-chi2min-lev

# compute where change in chi**2 = 2.706 for param[2]
chi2min = chi2_statistic(x, y, yerr, f2, params)
resp = root_scalar(dchi2, args=(2, 2.706, f2, chi2min), bracket=[params[2],params[2]+1], method='brentq')
resm = root_scalar(dchi2, args=(2, 2.706, f2, chi2min), bracket=[params[2],params[2]-1], method='brentq')

print('param[2] min, max: %6.3f, %6.3f' %(resm.root, resp.root))

# plot results
plt.figure(figsize=[10, 6])
plt.errorbar(x,y,yerr,ls='none', marker='o', label='data')
plt.plot(x, f2(x,params[0], params[1], resp.root), label='max')
plt.plot(x, f2(x,params[0], params[1], resm.root), label='min')
plt.xlabel('x')
plt.ylabel('$f(x)$')
plt.xlim([x_min, x_max])
plt.legend(fontsize=12);

This results is not that far off the esimate from the covariance matrix.

We can also compute the joint confidence as contour plots.  In this case the $\Delta \chi^2$ is larger because we are varying two parameters at the same time.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# construct contour plot ranges
# you may want to adjust these depending
# on your random realizations of the y errors
p1min, p1max = 1.2, 1.4
p2min, p2max = 0.2, 0.6

n = 200
p1 = np.linspace(p1min, p1max, n)
p2 = np.linspace(p2min, p2max, n)

# my pyplot functions (contour, pcolormesh, imshow, etc.) require
# 2D arrays for the x, y values.  This allows them to plot non-retangular
# grids.  But, we often just have two 1D arrays -- so NumPy provides
# meshgrid to turn two 1D arrays into two 2D arrays
P1, P2 = np.meshgrid(p1, p2)

# function for computing delta chi**2
def dchi2_2d(p1, p2, chi2min):
    p = np.copy(params)
    p[1] = p1
    p[2] = p2

    def ffixed(x, a):
        # keep it simple
        return f(x, a, p1, p2)

    # repeat fit with p1, p2 fixed
    p, pcov = curve_fit(ffixed, x, y, sigma=yerr)
    p = np.append(p, [p1,p2])
    # delta chi**2 = 2.706 is 90% confidence for one parameter
    return chi2_statistic(x, y, yerr, f2, p)-chi2min

chi2min = chi2_statistic(x, y, yerr, f2, params)
# loop over 2D grid -- this may take awhile
dchi = np.empty_like(P1)
for j in range(n):
    for i in range(n):
        dchi[j, i] = dchi2_2d(p1[i], p2[j], chi2min)

# plot contours corresponding to 68%, 90%, and 99% confidence
# based on delta chi**2
plt.contour(P1, P2, dchi, [2.3, 4.61, 9.21])
plt.xlabel('p[1]')
plt.ylabel('p[2]');

We find a modest anticorrelation in the best fit frequency (p[1]) and the best fit phase (p[2]).

The covariance matrix can also alert us to problems with our model. Lets add an extra parameter to our sinusoidal function.

In [ ]:
# fit with old function
f2 = lambda x, a, b, c: a*np.sin(b*x+c)  
params, pcov, info, msg, ierr = curve_fit(f2, x, y, full_output=True)
print("Covariance matrix: ",pcov)

# fit with new function
f3 = lambda x, a, b, c, d: a*np.sin(b*x+c+d) 
params, pcov = curve_fit(f3, x, y)
print("Covariance matrix: ",pcov)

The covariance matrix for the first function is pretty well behaved.  The first parameter representing the amplitude doesn't seem strongly correlated with the other parameters.  The last two parameters (essentially the frequency and phase) are slightly anticorrelated (as our contour plot above also shows).

In the second case, the much larger variance on the last two parameters (large diagonal values) and the large anticorrelation (large off diagonal value) tell us that these parameters are degenerate with each other, which is obvious by looking at `f3()`.  This case is obviously contrived but more subtle cases can arise in practice.

## 3. Non-linear Least Squares Fitting in More Detail

In this section, we dig in a little further to the default method used by Scipy's `curvefit()` function. This is the Levenberg-Marquardt (LM) method. Let's first begin by connecting to the optimization problems we studied last week.

### Newton's Method of Optimization

Last week we discussed Newton-Raphson (sometimes just Newton's method) as a method for curve fitting. It can also be used to find the minimum of a function i.e. Newton's method of optimization.

To derive it, we begin with a Taylor expansion to second order:
\begin{equation}
f(x) \approx f(x_0) + f'(x_0)(x - x_0) + (1/2)f''(x_0)(x - x_0)^2
\end{equation}

Now the minimum is at $f'(x) = 0$. If we take the derivative Taylor expansion, we have
\begin{equation}
f'(x) = f'(x_0) + f''(x_0)(x-x_0) = 0.
\end{equation}

So, if we have a guess for the minium $x_0$, we can construct a new guess by solving this for $x=x_1$, or more generally, given $x_m$, we can find $x_{m+1}$ via
\begin{equation}
x_{m+1} = x_m - f'(x_m)/f''(x_m)
\end{equation}

### Newton's Method in Multiple Dimensions

We now generalize to multiple dimensions. In constrast to the method of steepest descents, which we discussed last week, we will consider an alternative that is more of a generalization of Newton's method.

To begin let us consider a general function form
\begin{equation}
f(\mathbf{x}) = \sum_{i=0}^{N-1} r_i(\mathbf{x})^2.
\end{equation}
For example, you can think of $r_i$ as the error between a data point and a function with parameters $\mathbf{x} = (x_0, . . . ., x_{M-1})$ that is meant the fit the data.  Note that $M\neq N$: $M$ is the number of parameters, while $N$ is the number of data points to be fit. 

Given this function, we want to minimize it. Following the logic of Newton's method, lets Taylor expand out to second order. 
\begin{equation}
f(\mathbf{x}+\mathbf{dx}) \approx f(\mathbf{x}) + \sum_{j=0}^{M-1} \sum_{i=0}^{N-1} 2r_i(\mathbf{x})\frac{\partial r_i(\vec{x})}{\partial x_j}dx_j + \sum_{j=0}^{M-1} \sum_{k=0}^{M-1} \sum_{i=0}^{N-1} \left(\frac{\partial r_i(\mathbf{x})}{\partial x_k}\frac{\partial r_i(\mathbf{x})}{\partial x_j}+r_i(\mathbf{x})\frac{\partial^2 r_i(\mathbf{x})}{\partial x_j\partial x_k}\right)dx_jdx_k ,
\end{equation}
where the higher order terms are order $(\mathbf{dx}^3)$.

As a simplification, we drop the second derivative term but keep the product of the derivatives and find:
\begin{equation}
f(\mathbf{x}+\vec{dx}) \approx f(\vec{x}) + \sum_{j=0}^{M-1} \sum_{i=0}^{N-1}\left( 2r_i(\mathbf{x})\frac{\partial r_i(\mathbf{x})}{\partial x_j} +  \sum_{k=0}^{M-1} \frac{\partial r_i(\mathbf{x})}{\partial x_k}\frac{\partial r_i(\mathbf{x})}{\partial x_j}dx_k\right)dx_j
\end{equation}

We can rewrite this in matrix notation where ${\sf J} = J_{ij} = dr_i/dx_j$ is an $N\times M$ matrix and $\mathbf{r} = r_i(\mathbf{x})$ is a vector of length N. We then have:
\begin{equation}
f(\mathbf{x}+\mathbf{dx}) \approx f(\mathbf{x}) + 2\mathbf{dx}\cdot{\sf J}^T\cdot\mathbf{r} + \mathbf{dx}\cdot{\sf J}^T\cdot{\sf J}\cdot\mathbf{dx},
\end{equation}
where $^T$ means transpose.

As in Newton's method, we now take a gradient with respect to $\mathbf{dx}$ and set this equal to zero, to find
\begin{equation}
\nabla f(\vec{x}+\vec{dx}) = 0 
\end{equation}
so that
\begin{equation}
\mathbf{r}\cdot{\sf J} + {\sf J}^T\cdot{\sf J}\cdot\mathbf{dx} = 0
\end{equation}
 
Solving for $\mathbf{dx}$, we have
\begin{equation}
\mathbf{dx} = -({\sf J}^T\cdot {\sf J})^{-1}{\sf J}^T\mathbf{r}, 
\end{equation}
where $({\sf J}^T\cdot{\sf J})^{-1}$ is the inverse matrix of ${\sf J}^T\cdot{\sf J}$.

This may seem somewhat complicated to parse but it is relatively simple. The only thing that is different is rather than the $\delta x = -f(x)/f'(x)$ in Newton's is that we have a vector of ``$\mathbf{\delta x}$s'' that is set by a matrix equation.

### Gradient Descent Review

Recall last week we considered the case where we simply computed the gradient function along the different direction to update the vector $\mathbf{x}$ e.g.
\begin{equation}
\mathbf{x}' = \mathbf{x} - \nabla f(\mathbf{x}) \Delta x,
\end{equation}
where $\Delta x$ is our step parameter.

Using the function and notation used above, this can be written as
\begin{equation}
\mathbf{x}' = \mathbf{x} - \Delta x 2 {\sf J}^T \mathbf{r}.
\end{equation}

### Levenberg-Marquardt (LM) Algorithm

The generalized Newton's method generally converges faster near a minimum, but it can fail if the starting point is too far from the solution and/or the initial ${\sf J}$ matrix is poorly conditioned. The gradient descent method always goes downhill (will at least find a local minimum) but can be slow, particularly near the minimum. the LM method effectively interpolates between the two.

We choose our trial step via
\begin{equation}
\mathbf{h} = -({\sf J}^T \cdot {\sf J} + \lambda{\sf I})^{-1} {\sf J}^T \mathbf{r}
\end{equation}
where $\lambda$ is the damping parameter ($\lambda \geq 0$) and ${\sf I}$ is the identity matrix. When $\lambda$ is sufficiently large, the method is similar to gradient descent, but when $\lambda$ is small we recover the generalized Newton's method. This comes with a number of advantages: (1) by varying $\lambda$ we can use the method that is faster near or far from the minimum; (2) the addition of $\lambda {\sf I}$ to ${\sf J}^T \cdot {\sf J}$ avoid issues when the latter is nearly singular; and (3) will converge far from the solution.

A single iteration $k$ consists of the follwoing sub-steps:
1) Compute residuals: $\mathbf{r} = \mathbf{r}(\mathbf{x_k})$
2) Compute the Jacobian: $\mathbf{J} = \mathbf{J}(\mathbf{x_k})$
3) Solve: $({\sf J}^T\cdot {\sf J} + \lambda_k{\sf I})\mathbf{h} = -{\sf J}^T\mathbf{r}$
4) Compute trial point: $\mathbf{x}_{\rm trial} = \mathbf{x_k} + \mathbf{h}$
5) Compute new residuals: $\mathbf{r}_{\rm trial} = \mathbf{r}(\mathbf{x}_{\rm trial})$
6) Compute gain ratio:
\begin{equation}
\rho = \frac{\|\mathbf{r}\|^2 - \|\mathbf{r}_{\text{trial}}\|^2}{- 2\mathbf{h}^T {\sf J}^T \mathbf{r} + \mathbf{h}^T {\sf J}^T\cdot{\sf J} }
\end{equation}        
7) Update:
\begin{equation}
\text{If } \rho > 0: \quad \mathbf{x}_{k+1} = \mathbf{x}_{\rm trial}, \quad \lambda_{k+1} = \lambda_k / \nu
\end{equation}
\begin{equation}
\text{If } \rho \leq 0: \quad \mathbf{x}_{k+1} = \mathbf{x}_{k}, \quad \lambda_{k+1} = \lambda_k \times \nu
\end{equation}
where $\nu > 1$. In other words, if $\rho \le 0$, there is no improvement so we reject our trial and increase $\lambda$.


### Beyond Least Squares

Although least squares is one of the most popular fit statistics, it is not always appropriate under all conditions.  Strictly speaking, least squares is only valid when the errors on the $x_i$ are negligible (although the method can be generalized to handle this case) and when the errors on $y_i$ are normally distributed.

As a theorist, I rarely have to worry about this.  But, if you want to make a living crunching data, you should invest time in understanding the limits of least squares fitting and thinking about more general concepts, such as maximum likelihood estimation and how that applies to your data analysis requirements.